# Checkpoint 2: Pneumonia Detection from Chest X-Rays

Binary image classification - **NORMAL** vs **PNEUMONIA** chest X-rays.

**Dataset:** [Kaggle - Chest X-Ray Images (Pneumonia)](https://www.kaggle.com/datasets/paultimothymooney/chest-xray-pneumonia) by Paul Mooney

**Sections:**
1. Dataset download & setup
2. Data exploration and visualisation
3. Preprocessing and augmentation
4. Baseline CNN model training
5. Evaluation and metrics

> **Colab tip:** Go to **Runtime -> Change runtime type -> T4 GPU** for faster training.

## How to Run This Notebook

### Local setup in VS Code
Run these commands once in a PowerShell terminal from the project folder:
```powershell
python -m venv .venv
.\.venv\Scripts\Activate.ps1
python -m pip install -U pip
python -m pip install tensorflow numpy pandas matplotlib scikit-learn seaborn pillow kaggle
```
Then select `.venv` as the notebook kernel in VS Code. After that, you can skip the dependency install cell unless packages are missing or changed.

For Kaggle locally, either set `KAGGLE_API_TOKEN` before launching VS Code/Jupyter, or put your token in `<your-home-folder>\.kaggle\access_token` with no `.txt` extension. On Windows this is usually `C:\Users\<your-username>\.kaggle\access_token`.

To set the environment variable for the current PowerShell session only:
```powershell
$env:KAGGLE_API_TOKEN = "paste-your-token-here"
```

To save it permanently for your Windows user account, run this once, then restart VS Code so the notebook kernel inherits it:
```powershell
[Environment]::SetEnvironmentVariable("KAGGLE_API_TOKEN", "paste-your-token-here", "User")
```

### Google Colab setup
Run the dependency install cell each time you start a fresh Colab runtime.

For Kaggle in Colab, save your token text once in Google Drive at `MyDrive/secrets/kaggle_token.txt`. The dataset setup cell below will mount Drive and load it.

### Shared cells
The dependency and dataset setup cells are written to work in both local VS Code and Google Colab. They follow the active notebook kernel, so local runs use `.venv` when that kernel is selected, while Colab runs use the Colab runtime.

In [3]:
import subprocess
import sys

DEPENDENCIES = [
    'tensorflow',
    'numpy',
    'pandas',
    'matplotlib',
    'scikit-learn',
    'seaborn',
    'pillow',
    'kaggle',
]

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', *DEPENDENCIES],
    check=True,
)

print(f'Dependencies installed for: {sys.executable}')


Dependencies installed for: c:\Users\lil\Documents\Adv_AI_Project_Pneumonia\.venv\Scripts\python.exe


## 0 - Dataset Setup

This section downloads the Chest X-Ray Pneumonia dataset with the Kaggle API. Use the credential setup from **How to Run This Notebook** above before running the cell.

### Local VS Code
- The cell accepts either `KAGGLE_API_TOKEN` or `<your-home-folder>\.kaggle\access_token`.
- If you set `KAGGLE_API_TOKEN`, restart VS Code/Jupyter before running the notebook so the kernel can see it.

### Google Colab
- Expected token file in Drive: `MyDrive/secrets/kaggle_token.txt`
- The next cell mounts Drive, reads that token, and stores it in `KAGGLE_API_TOKEN` for the current runtime.

The notebook expects the dataset at `chest_xray/train...`, `chest_xray/val...`, and `chest_xray/test...`. After download it removes Kaggle archive clutter like `chest_xray/chest_xray` and `chest_xray/__MACOSX`, then prints image counts.

In [ ]:
import os
import shutil
import sys
from pathlib import Path

KAGGLE_DATASET = 'paultimothymooney/chest-xray-pneumonia'
KAGGLE_TOKEN_ENV = 'KAGGLE_API_TOKEN'
COLAB_TOKEN_PATH = Path('/content/drive/MyDrive/secrets/kaggle_token.txt')
KAGGLE_DIR = Path.home() / '.kaggle'
EXPECTED_SPLITS = ('train', 'val', 'test')
EXPECTED_CLASSES = ('NORMAL', 'PNEUMONIA')
VALID_EXTENSIONS = {'.jpeg', '.jpg', '.png', '.bmp'}

try:
    from google.colab import drive
    RUNNING_IN_COLAB = True
except ImportError:
    RUNNING_IN_COLAB = False

DOWNLOAD_DIR = Path('/content') if RUNNING_IN_COLAB else Path.cwd()
DATASET_DIR = DOWNLOAD_DIR / 'chest_xray'

if RUNNING_IN_COLAB:
    drive.mount('/content/drive')
    if not os.environ.get(KAGGLE_TOKEN_ENV):
        if not COLAB_TOKEN_PATH.exists():
            raise FileNotFoundError(f'Kaggle token not found at {COLAB_TOKEN_PATH}')
        os.environ[KAGGLE_TOKEN_ENV] = COLAB_TOKEN_PATH.read_text(encoding='utf-8').strip()
    auth_source = f'Colab/Drive token: {COLAB_TOKEN_PATH}'
elif os.environ.get(KAGGLE_TOKEN_ENV):
    auth_source = f'environment variable: {KAGGLE_TOKEN_ENV}'
elif (KAGGLE_DIR / 'access_token').exists():
    token_path = KAGGLE_DIR / 'access_token'
    os.environ[KAGGLE_TOKEN_ENV] = token_path.read_text(encoding='utf-8').strip()
    auth_source = f'token file: {token_path}'
else:
    raise EnvironmentError(
        f'No Kaggle credentials found. Set {KAGGLE_TOKEN_ENV}, '
        f'or create {KAGGLE_DIR / "access_token"}.'
    )

active_token = os.environ.get(KAGGLE_TOKEN_ENV, '')
token_preview = f'{active_token[:4]}...{active_token[-4:]}'
print(f'Kaggle auth source: {auth_source}')
print(f'Kaggle token preview: {token_preview}')

from kaggle.api.kaggle_api_extended import KaggleApi

print(f'Downloading/extracting {KAGGLE_DATASET} into {DOWNLOAD_DIR}')
api = KaggleApi()
api.authenticate()
api.dataset_download_files(KAGGLE_DATASET, path=str(DOWNLOAD_DIR), unzip=True, quiet=False)

missing = [DATASET_DIR / split / cls
           for split in EXPECTED_SPLITS for cls in EXPECTED_CLASSES
           if not (DATASET_DIR / split / cls).is_dir()]
if missing:
    raise FileNotFoundError('Missing expected dataset folders: ' + ', '.join(str(path) for path in missing))

for extra_path in (DATASET_DIR / 'chest_xray', DATASET_DIR / '__MACOSX'):
    if extra_path.exists():
        shutil.rmtree(extra_path)
        print(f'Removed archive clutter: {extra_path}')

print(f'Dataset root: {DATASET_DIR}')
for split in EXPECTED_SPLITS:
    counts = {}
    for cls in EXPECTED_CLASSES:
        folder = DATASET_DIR / split / cls
        counts[cls] = sum(1 for file in folder.iterdir() if file.suffix.lower() in VALID_EXTENSIONS)
    total = sum(counts.values())
    print(f'{split:5s} total={total:4d} | NORMAL={counts["NORMAL"]:4d} | PNEUMONIA={counts["PNEUMONIA"]:4d}')

Kaggle auth source: environment variable: KAGGLE_API_TOKEN
Kaggle token preview: KGAT...8e53
Downloading/extracting paultimothymooney/chest-xray-pneumonia into c:\Users\lil\Documents\Adv_AI_Project_Pneumonia
Dataset URL: https://www.kaggle.com/datasets/paultimothymooney/chest-xray-pneumonia


100%|██████████| 2.29G/2.29G [08:07<00:00, 5.06MB/s]


## Configuration

All paths and hyperparameters are defined here. Edit `EPOCHS` or `BATCH_SIZE` if needed.

In [ ]:
import sys
from pathlib import Path

# Paths
BASE_DIR = Path('/content') if 'google.colab' in sys.modules else Path.cwd()
DATASET_DIR = BASE_DIR / 'chest_xray'
OUTPUT_DIR  = BASE_DIR / 'outputs'
FIGURES_DIR = OUTPUT_DIR / 'figures'
RESULTS_DIR = OUTPUT_DIR / 'results'
MODELS_DIR  = OUTPUT_DIR / 'models'

EXPECTED_SPLITS  = ('train', 'val', 'test')
EXPECTED_CLASSES = ('NORMAL', 'PNEUMONIA')
VALID_EXTENSIONS = {'.jpeg', '.jpg', '.png', '.bmp'}

# Hyperparameters
IMAGE_SIZE  = (224, 224)
BATCH_SIZE  = 32
EPOCHS      = 10
RANDOM_SEED = 42

for d in (FIGURES_DIR, RESULTS_DIR, MODELS_DIR):
    d.mkdir(parents=True, exist_ok=True)

print('Configuration loaded.')
print(f'  Dataset : {DATASET_DIR}')
print(f'  Outputs : {OUTPUT_DIR}')
print(f'  Image size={IMAGE_SIZE}  Batch size={BATCH_SIZE}  Epochs={EPOCHS}')

## 1 · Data Exploration

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from PIL import Image

def _list_images(directory):
    return sorted(
        f for f in directory.iterdir()
        if f.is_file() and f.suffix.lower() in VALID_EXTENSIONS
    )

# Verify dataset structure
if not DATASET_DIR.exists():
    raise FileNotFoundError(f'Dataset not found at {DATASET_DIR}. Run the download cell first.')

missing = [DATASET_DIR / s / c
           for s in EXPECTED_SPLITS for c in EXPECTED_CLASSES
           if not (DATASET_DIR / s / c).exists()]
if missing:
    print('WARNING - missing folders:')
    for p in missing:
        print(f'  {p}')
else:
    print('Dataset structure OK')


# Count images per split/class
rows = [{'split': s, 'class': c, 'count': len(_list_images(DATASET_DIR / s / c))}
        for s in EXPECTED_SPLITS for c in EXPECTED_CLASSES]
summary_df = pd.DataFrame(rows)
summary_df.to_csv(RESULTS_DIR / 'dataset_summary.csv', index=False)
print(summary_df.pivot(index='split', columns='class', values='count').fillna(0).astype(int))

# Class distribution bar plot
plt.figure(figsize=(8, 5))
sns.barplot(data=summary_df, x='split', y='count', hue='class')
plt.title('Class Distribution by Dataset Split')
plt.ylabel('Number of Images')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'class_distribution.png', dpi=150)
plt.show()

# Sample image grid
MAX_PER_CLASS = 4
fig, axes = plt.subplots(2, MAX_PER_CLASS, figsize=(3 * MAX_PER_CLASS, 6), squeeze=False)
for row_idx, cls in enumerate(EXPECTED_CLASSES):
    samples = _list_images(DATASET_DIR / 'train' / cls)[:MAX_PER_CLASS]
    for col_idx in range(MAX_PER_CLASS):
        ax = axes[row_idx][col_idx]
        if col_idx < len(samples):
            ax.imshow(Image.open(samples[col_idx]).convert('L'), cmap='gray')
            ax.set_title(cls, fontsize=9)
        ax.axis('off')
plt.suptitle('Sample Training Images', y=1.01)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'example_images_grid.png', dpi=150)
plt.show()

## 2 · Preprocessing & Data Augmentation

Each image is resized to **224x224**, normalised to **[0, 1]**, and converted from grayscale to 3 channels.  
Training images are augmented with random rotation (5%), zoom (10%), horizontal flip, and contrast shift (10%).

In [ ]:
import tensorflow as tf

AUTOTUNE = tf.data.AUTOTUNE

def load_images_from_directory(directory, shuffle=True):
    return tf.keras.utils.image_dataset_from_directory(
        directory,
        labels='inferred',
        label_mode='binary',
        color_mode='grayscale',
        image_size=IMAGE_SIZE,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        seed=RANDOM_SEED,
    )

def resize_and_normalize(images):
    return tf.cast(images, tf.float32) / 255.0

def ensure_three_channel(images):
    if images.shape[-1] == 1:
        return tf.image.grayscale_to_rgb(images)
    if images.shape[-1] > 3:
        return images[..., :3]
    return images

def get_augmentation_layer():
    return tf.keras.Sequential([
        tf.keras.layers.RandomRotation(0.05),
        tf.keras.layers.RandomZoom(0.1),
        tf.keras.layers.RandomFlip('horizontal'),
        tf.keras.layers.RandomContrast(0.1),
    ], name='augmentation')

def preprocess_dataset(dataset, augment=False, augmentation_layer=None):
    def _preprocess(images, labels):
        images = resize_and_normalize(images)
        images = ensure_three_channel(images)
        if augment and augmentation_layer is not None:
            images = augmentation_layer(images, training=True)
        return images, labels
    return dataset.map(_preprocess, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)

def create_datasets():
    train_raw = load_images_from_directory(DATASET_DIR / 'train', shuffle=True)
    val_raw   = load_images_from_directory(DATASET_DIR / 'val',   shuffle=False)
    test_raw  = load_images_from_directory(DATASET_DIR / 'test',  shuffle=False)
    class_names = train_raw.class_names
    aug_layer   = get_augmentation_layer()
    train_ds = preprocess_dataset(train_raw, augment=True, augmentation_layer=aug_layer)
    val_ds   = preprocess_dataset(val_raw)
    test_ds  = preprocess_dataset(test_raw)
    return train_ds, val_ds, test_ds, class_names

print('Preprocessing functions ready.')


## 3 · Baseline CNN Model

3 x (Conv2D + MaxPooling) blocks with 32/64/128 filters, followed by Flatten, Dropout(0.3), Dense(128), Dropout(0.2), and a sigmoid output.  
Compiled with **Adam** optimizer, **binary crossentropy** loss, and metrics: accuracy, precision, recall.

In [ ]:
def build_baseline_model(input_shape=(IMAGE_SIZE[0], IMAGE_SIZE[1], 3)):
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=input_shape),
        tf.keras.layers.Conv2D(32, (3, 3), activation='relu'),
        tf.keras.layers.MaxPooling2D((2, 2)),
        tf.keras.layers.Conv2D(64, (3, 3), activation='relu'),
        tf.keras.layers.MaxPooling2D((2, 2)),
        tf.keras.layers.Conv2D(128, (3, 3), activation='relu'),
        tf.keras.layers.MaxPooling2D((2, 2)),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(1, activation='sigmoid'),
    ])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(),
        loss='binary_crossentropy',
        metrics=[
            tf.keras.metrics.BinaryAccuracy(name='accuracy'),
            tf.keras.metrics.Precision(name='precision'),
            tf.keras.metrics.Recall(name='recall'),
        ],
    )
    return model

model = build_baseline_model()
model.summary()


## 4 · Training

Early stopping monitors `val_loss` with patience=3 and restores the best weights automatically.

In [ ]:
import numpy as np

tf.random.set_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

train_ds, val_ds, test_ds, class_names = create_datasets()
print(f'Class index mapping: {class_names}')

model = build_baseline_model()

early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True,
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=[early_stopping],
    verbose=1,
)


In [ ]:
# Save training history CSV
history_df = pd.DataFrame(history.history)
history_df.to_csv(RESULTS_DIR / 'training_history.csv', index=False)

# Plot accuracy and loss curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['accuracy'],     label='Train')
axes[0].plot(history.history['val_accuracy'], label='Val')
axes[0].set_title('Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot(history.history['loss'],     label='Train')
axes[1].plot(history.history['val_loss'], label='Val')
axes[1].set_title('Loss')
axes[1].set_xlabel('Epoch')
axes[1].legend()

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'training_curves.png', dpi=150)
plt.show()

# Save model
model.save(MODELS_DIR / 'baseline_cnn.keras')
print(f'Model saved to {MODELS_DIR}/baseline_cnn.keras')


## 5 · Evaluation

In [ ]:
from sklearn.metrics import confusion_matrix

# Evaluate on test set
test_results = model.evaluate(test_ds, verbose=1)
metric_names = ['loss', 'accuracy', 'precision', 'recall']
metrics      = dict(zip(metric_names, test_results))

print('Test Metrics:')
for k, v in metrics.items():
    print(f'  {k}: {v:.4f}')

with open(RESULTS_DIR / 'baseline_metrics.txt', 'w') as f:
    for k, v in metrics.items():
        print(f'{k}: {v:.4f}', file=f)

# Collect predictions for confusion matrix
y_true, y_pred = [], []
for images, labels in test_ds:
    probs = model.predict(images, verbose=0).flatten()
    preds = (probs >= 0.5).astype(int)
    y_true.extend(labels.numpy().flatten().astype(int))
    y_pred.extend(preds)

cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix - Baseline CNN')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'confusion_matrix.png', dpi=150)
plt.show()

print(f'All outputs saved to {OUTPUT_DIR}')
